# Reproducibility notebook — *Do Text-to-Music Models Take Direction?*
Regenerates every banked statistic, table, and figure in the paper from the released CSVs (Tier 1: CPU only, minutes, no audio, no DEAM).

Run from anywhere inside the repository, for example from the repo root:
```
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output-dir <dir> notebooks/repro.ipynb
```
Paths (environment variables; defaults in brackets):
- `REPRO_DATA_DIR` — input CSVs [`<repo>/data`]. Read only; never written.
- `REPRO_OUT_DIR` — outputs [`<repo>/repro_out`]: regenerated banks in `tables/` (same file names as `data/`, so they diff cell by cell), figures in `figures/`. The committed `figures/` folder is never overwritten unless `REPRO_ALLOW_COMMITTED_FIGURES=1`.

Inputs used:
```
data/scored_full_v2.csv        # per-clip scores, raw + loudness-normalized lenses
data/features_verbal.csv       # per-clip acoustic descriptors (verbal clips)
data/arousal_index_ridge.json  # fitted Arousal Index ridge (scripts/fit_arousal_ridge.py)
data/s7_clap_axis.csv          # per-clip fixed-anchor CLAP axis (Tier-2 output)
data/s6_*.csv, data/s7_*.csv   # banked outputs: the cross-check targets
```
All bootstrap confidence intervals: 2,000 clip-level resamples, percentile method, seed 0 per call. The banked-statistics cells (S6, S7) are ported from `notebooks/scoring.ipynb` cells 13–14 and `notebooks/clap-axis.ipynb` with their statistical logic unchanged.

In [1]:

import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

B, SEED = 2000, 0
def _find_repo_root(start):
    p = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(p, "data", "scored_full_v2.csv")):
            return p
        if os.path.dirname(p) == p:
            return os.path.abspath(start)
        p = os.path.dirname(p)
ROOT   = _find_repo_root(os.getcwd())
DATA   = os.path.abspath(os.environ.get("REPRO_DATA_DIR", os.path.join(ROOT, "data")))
OUT    = os.path.abspath(os.environ.get("REPRO_OUT_DIR", os.path.join(ROOT, "repro_out")))
FIGS   = os.path.join(OUT, "figures")
TABLES = os.path.join(OUT, "tables")
_committed_figs = os.path.realpath(os.path.join(ROOT, "figures"))
assert os.path.realpath(FIGS) != _committed_figs or os.environ.get("REPRO_ALLOW_COMMITTED_FIGURES") == "1", \
    "refusing to overwrite the committed figures/ (set REPRO_ALLOW_COMMITTED_FIGURES=1 to allow)"
assert os.path.realpath(TABLES) != os.path.realpath(DATA), "refusing to write banks into the data folder"
os.makedirs(FIGS, exist_ok=True); os.makedirs(TABLES, exist_ok=True)
rel = lambda p: os.path.relpath(p, ROOT)   # for printing only
print(f"data: {rel(DATA)}\nout:  {rel(OUT)}")

df = pd.read_csv(f"{DATA}/scored_full_v2.csv")
assert len(df) == 1500 and df.file.is_unique
SV = df[df.phrasing == "verbal"]
MODELS = sorted(df.model.unique())
LEVELS = [1, 3, 5, 7, 9]

def opt(path):
    return pd.read_csv(path) if os.path.exists(path) else None
cis_bank  = opt(f"{DATA}/s6_bootstrap_cis.csv")
decomp    = opt(f"{DATA}/s6_feature_decomp.csv")
raw_lvl   = opt(f"{DATA}/s6_raw_ridge_level_means.csv")
feats     = opt(f"{DATA}/features_verbal.csv")
s7_cis    = opt(f"{DATA}/s7_bootstrap_cis.csv")

def ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

def content_share(xs):
    """K4 content share for one resample: normalized delta / raw delta (L9-L1).
    NaN when the raw delta is ~0 (|r| <= 1e-9), as in the canonical scoring run;
    ci() then drops it. Scores are quantized to 0.1, so exact-zero deltas occur."""
    r = xs[1][:, 0].mean() - xs[0][:, 0].mean()
    return (xs[1][:, 1].mean() - xs[0][:, 1].mean()) / r if abs(r) > 1e-9 else np.nan

def boot_rho(lv, sc):
    rng = np.random.default_rng(SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([spearmanr(lv[i], sc[i])[0] for i in idx])

def boot_stat(arrs, fn):
    rng = np.random.default_rng(SEED)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    return np.array([fn([a[ix[b]] for a, ix in zip(arrs, idxs)]) for b in range(B)])

print(f"loaded {len(df)} clips | models: {MODELS}")


data: data
out:  rerun/step1/attempt2/out
loaded 1500 clips | models: ['ace', 'mgm', 'mgs', 'sa3s', 'sao']


In [2]:

# ---- Table 1: monotonicity, phrasing contrast, dynamic range (median + mean) ----
rows = []
for m in MODELS:
    r = {}
    for ph in ["numeric", "verbal"]:
        g = df[(df.model == m) & (df.phrasing == ph)]
        rho = spearmanr(g.level, g.p0a)[0]
        lo, hi = ci(boot_rho(g.level.values, g.p0a.values))
        r[f"rho_{ph}"] = f"{rho:+.3f} [{lo:+.3f},{hi:+.3f}]"
        if ph == "verbal":
            a1 = g[g.level == 1].p0a.values; a9 = g[g.level == 9].p0a.values
            md = np.median(a9) - np.median(a1)
            mlo, mhi = ci(boot_stat([a1, a9], lambda xs: np.median(xs[1]) - np.median(xs[0])))
            mn = a9.mean() - a1.mean()
            nlo, nhi = ci(boot_stat([a1, a9], lambda xs: xs[1].mean() - xs[0].mean()))
            r["median_delta"] = f"{md:+.2f} [{mlo:+.2f},{mhi:+.2f}]"
            r["mean_delta"] = f"{mn:+.2f} [{nlo:+.2f},{nhi:+.2f}]"
    gn = df[(df.model == m) & (df.phrasing == "numeric")]
    gv = df[(df.model == m) & (df.phrasing == "verbal")]
    drho = spearmanr(gv.level, gv.p0a)[0] - spearmanr(gn.level, gn.p0a)[0]
    dlo, dhi = ci(boot_rho(gv.level.values, gv.p0a.values) - boot_rho(gn.level.values, gn.p0a.values))
    r["delta_rho_vn"] = f"{drho:+.3f} [{dlo:+.3f},{dhi:+.3f}]"
    rows.append(dict(model=m, **r))
t1 = pd.DataFrame(rows).set_index("model")
print("TABLE 1 (paper §4.1)"); print(t1.to_string())

if cis_bank is not None:   # cross-check against the banked canonical values
    chk = cis_bank[(cis_bank.analysis == "rho_level_p0a") & (cis_bank.scope == "verbal")]
    for _, row in chk.iterrows():
        g = df[(df.model == row.model) & (df.phrasing == "verbal")]
        assert abs(spearmanr(g.level, g.p0a)[0] - row.point) < 1e-3
    print("\ncross-check vs s6_bootstrap_cis.csv: PASS")


TABLE 1 (paper §4.1)
                  rho_numeric              rho_verbal         median_delta           mean_delta            delta_rho_vn
model                                                                                                                  
ace    +0.070 [-0.089,+0.228]  +0.328 [+0.167,+0.476]  +0.10 [-0.05,+0.70]  +0.18 [-0.12,+0.47]  +0.258 [+0.063,+0.451]
mgm    +0.038 [-0.134,+0.202]  +0.538 [+0.414,+0.644]  +1.30 [+0.50,+2.10]  +1.28 [+0.91,+1.63]  +0.500 [+0.333,+0.668]
mgs    +0.041 [-0.127,+0.197]  +0.388 [+0.247,+0.512]  +2.05 [+0.80,+2.40]  +0.93 [+0.40,+1.45]  +0.347 [+0.199,+0.497]
sa3s   +0.035 [-0.139,+0.198]  +0.384 [+0.223,+0.525]  +1.60 [+0.55,+2.20]  +0.83 [+0.31,+1.33]  +0.350 [+0.222,+0.482]
sao    +0.007 [-0.165,+0.164]  +0.504 [+0.367,+0.619]  +1.30 [+0.55,+2.00]  +1.13 [+0.78,+1.50]  +0.497 [+0.370,+0.627]

cross-check vs s6_bootstrap_cis.csv: PASS


In [3]:

# ---- Table 2: mechanism (content share CIs recomputed; knob shares from bank) ----
print("TABLE 2 (paper §4.2)")
for m in MODELS:
    g = SV[SV.model == m]
    A1 = g[g.level == 1][["p0a", "p0a_n"]].values
    A9 = g[g.level == 9][["p0a", "p0a_n"]].values
    raw_d = A9[:, 0].mean() - A1[:, 0].mean()
    rd_ci = ci(boot_stat([A1, A9], lambda xs: xs[1][:, 0].mean() - xs[0][:, 0].mean()))
    if rd_ci[0] <= 0 <= rd_ci[1]:
        print(f"  {m:5s} content share: undefined (raw-delta CI covers 0)")
        continue
    share = (A9[:, 1].mean() - A1[:, 1].mean()) / raw_d
    slo, shi = ci(boot_stat([A1, A9], content_share))
    print(f"  {m:5s} content share {share:+.2f} [{slo:+.2f},{shi:+.2f}]")
if decomp is not None:
    print("\nfeature-cluster shares of the pre-calibration delta (banked):")
    print(decomp[["model", "d_raw_total", "loudness_share", "noisiness_share",
                  "tonality_share"]].round(3).to_string(index=False))


TABLE 2 (paper §4.2)
  ace   content share: undefined (raw-delta CI covers 0)
  mgm   content share +0.76 [+0.63,+0.91]
  mgs   content share +0.76 [+0.47,+0.99]
  sa3s  content share +0.82 [+0.64,+1.15]
  sao   content share +0.59 [+0.50,+0.69]

feature-cluster shares of the pre-calibration delta (banked):
model  d_raw_total  loudness_share  noisiness_share  tonality_share
  ace        0.208           0.094            0.259           0.647
  mgm        1.857          -0.019            0.623           0.397
  mgs        0.965           0.207            0.266           0.527
 sa3s        0.728           0.183            0.391           0.426
  sao        1.389           0.384            0.380           0.236


In [4]:

# ---- Figure 1: compliance curves with 95% bootstrap bands ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)
colors = dict(zip(MODELS, plt.rcParams["axes.prop_cycle"].by_key()["color"]))
for ax, ph in zip(axes, ["numeric", "verbal"]):
    for m in MODELS:
        mu, lo, hi = [], [], []
        for L in LEVELS:
            a = df[(df.model == m) & (df.phrasing == ph) & (df.level == L)].p0a.values
            mu.append(a.mean())
            l, h = ci(boot_stat([a], lambda xs: xs[0].mean()))
            lo.append(l); hi.append(h)
        ax.plot(LEVELS, mu, "-o", ms=4, color=colors[m], label=m)
        ax.fill_between(LEVELS, lo, hi, color=colors[m], alpha=0.15, lw=0)
    ax.set_title(f"{ph} prompts"); ax.set_xlabel("requested energy level")
    ax.set_xticks(LEVELS); ax.grid(alpha=0.25)
axes[0].set_ylabel("Arousal Index (0–11)"); axes[1].legend(fontsize=8)
fig.suptitle("Compliance curves — mean with 95% clip-level bootstrap bands", y=1.02)
fig.tight_layout(); fig.savefig(f"{FIGS}/compliance_curves_ci.png", dpi=200, bbox_inches="tight")
print(f"wrote {rel(FIGS)}/compliance_curves_ci.png")


wrote rerun/step1/attempt2/out/figures/compliance_curves_ci.png


In [5]:

# ---- Per-genre verbal curves (5 x 5 grid) ----
genres = sorted(df.genre_slug.unique())
fig, axes = plt.subplots(len(MODELS), len(genres), figsize=(13, 10),
                         sharex=True, sharey=True)
for i, m in enumerate(MODELS):
    for j, g in enumerate(genres):
        sub = SV[(SV.model == m) & (SV.genre_slug == g)]
        mu = [sub[sub.level == L].p0a.mean() for L in LEVELS]
        axes[i, j].plot(LEVELS, mu, "-o", ms=3)
        if i == 0: axes[i, j].set_title(g, fontsize=9)
        if j == 0: axes[i, j].set_ylabel(m, fontsize=9)
        axes[i, j].grid(alpha=0.25)
fig.suptitle("Verbal compliance by model × genre (mean Arousal Index)")
fig.tight_layout(); fig.savefig(f"{FIGS}/per_genre_curves.png", dpi=160, bbox_inches="tight")
print(f"wrote {rel(FIGS)}/per_genre_curves.png")


wrote rerun/step1/attempt2/out/figures/per_genre_curves.png


In [6]:

# ---- Droop panel: three witnesses at the top of the scale (paper §4.4) ----
print("verbal L9 − L7, 95% CIs")
for m in MODELS:
    g7 = SV[(SV.model == m) & (SV.level == 7)]
    g9 = SV[(SV.model == m) & (SV.level == 9)]
    out = []
    for col, lab in [("p0a", "calibrated"), ("clap", "own-prompt CLAP")]:
        pt = g9[col].mean() - g7[col].mean()
        lo, hi = ci(boot_stat([g7[col].values, g9[col].values],
                              lambda xs: xs[1].mean() - xs[0].mean()))
        out.append(f"{lab} {pt:+.3f} [{lo:+.3f},{hi:+.3f}]")
    print(f"  {m:5s} " + " | ".join(out))
if feats is not None:
    print("\nraw (pre-calibration) witness: recomputable from data/features_verbal.csv")
elif raw_lvl is not None:
    r = raw_lvl.set_index("model")
    print("\nraw (pre-calibration) level means (banked; CIs in s6_bootstrap_cis.csv):")
    print(r.round(2).to_string())


verbal L9 − L7, 95% CIs
  ace   calibrated -0.553 [-0.823,-0.250] | own-prompt CLAP -0.081 [-0.132,-0.027]
  mgm   calibrated +0.433 [+0.150,+0.747] | own-prompt CLAP +0.093 [+0.015,+0.163]
  mgs   calibrated -0.420 [-0.857,+0.027] | own-prompt CLAP -0.046 [-0.120,+0.030]
  sa3s  calibrated -0.303 [-0.690,+0.057] | own-prompt CLAP -0.001 [-0.086,+0.085]
  sao   calibrated +0.233 [+0.030,+0.440] | own-prompt CLAP +0.067 [-0.000,+0.136]

raw (pre-calibration) witness: recomputable from data/features_verbal.csv


In [7]:

# ---- Plausibility lens: bands, edge cells, step-gain contrasts (paper §4.3) ----
BANDS = {"folk": (0.5, 5.5), "edm": (4, 8), "hiphop": (2, 7),
         "orchestral": (3, 7), "rock": (5, 8)}   # rock analyst-set (H1 v2)
STEPS = [(1, 3), (3, 5), (5, 7), (7, 9)]
inb = lambda g, L: BANDS[g][0] <= L <= BANDS[g][1]

def contrast(sub, genres):
    cells = {(g, L): gg.p0a.values
             for (g, L), gg in sub.groupby(["genre_slug", "level"]) if g in genres}
    keys = sorted(cells); arrs = [cells[k] for k in keys]
    pos = {k: i for i, k in enumerate(keys)}
    def gap(xs, which):
        gains = [(xs[pos[(g, b)]].mean() - xs[pos[(g, a)]].mean()) / (b - a)
                 for g in genres for a, b in STEPS
                 if (inb(g, a) and inb(g, b)) == which and (g, a) in pos]
        return float(np.mean(gains))
    pt = gap(arrs, True) - gap(arrs, False)
    lo, hi = ci(boot_stat(arrs, lambda xs: gap(xs, True) - gap(xs, False)))
    return pt, lo, hi

for label, genres in [("all genres", list(BANDS)), ("rock excluded", [g for g in BANDS if g != "rock"])]:
    pt, lo, hi = contrast(SV, genres)
    print(f"pooled in-band minus out-of-band gain, {label}: {pt:+.3f} [{lo:+.3f},{hi:+.3f}]")

edge = SV.groupby(["model", "genre_slug", "level"]).p0a.mean().round(2).unstack()
print("\nper-cell verbal means (see paper App. A; '*' marking is printed by S6.5 below):")
print(edge.to_string())


pooled in-band minus out-of-band gain, all genres: +0.190 [+0.096,+0.279]
pooled in-band minus out-of-band gain, rock excluded: +0.217 [+0.111,+0.321]

per-cell verbal means (see paper App. A; '*' marking is printed by S6.5 below):
level                1     3     5     7     9
model genre_slug                              
ace   edm         4.65  4.28  4.48  5.20  4.38
      folk        3.77  3.77  4.02  4.62  3.87
      hiphop      4.57  3.87  4.37  5.10  4.35
      orchestral  3.70  3.90  4.10  4.40  4.05
      rock        4.13  4.20  4.67  5.18  5.08
mgm   edm         5.87  6.38  6.18  6.32  6.50
      folk        4.43  4.03  5.82  4.90  6.25
      hiphop      5.38  4.65  6.32  6.33  6.50
      orchestral  3.77  4.67  4.97  5.98  6.47
      rock        6.37  6.38  6.48  6.50  6.48
mgs   edm         5.40  5.73  6.47  6.42  6.38
      folk        4.15  4.08  4.73  5.72  5.00
      hiphop      4.98  5.53  6.18  6.28  6.33
      orchestral  3.90  3.97  4.02  5.67  4.72
      rock      

In [8]:

# ---- Skew exhibit: clip-level scores at L1 vs L9 (why median > mean for mgs) ----
fig, axes = plt.subplots(1, len(MODELS), figsize=(13, 3.2), sharey=True)
rng = np.random.default_rng(0)
for ax, m in zip(axes, MODELS):
    for L, x0 in [(1, 0), (9, 1)]:
        a = SV[(SV.model == m) & (SV.level == L)].p0a.values
        ax.scatter(x0 + rng.uniform(-0.12, 0.12, len(a)), a, s=10, alpha=0.6)
        ax.hlines([np.median(a), a.mean()], x0 - 0.2, x0 + 0.2,
                  colors=["k", "r"], lw=[1.6, 1.0])
    ax.set_xticks([0, 1]); ax.set_xticklabels(["L1", "L9"]); ax.set_title(m)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("Arousal Index")
fig.suptitle("Clip-level scores, verbal L1 vs L9 (black = median, red = mean)", y=1.04)
fig.tight_layout(); fig.savefig(f"{FIGS}/skew_L1L9.png", dpi=160, bbox_inches="tight")
print(f"wrote {rel(FIGS)}/skew_L1L9.png")


wrote rerun/step1/attempt2/out/figures/skew_L1L9.png


In [9]:

# ---- Leaderboard (banked) + cross-check against recomputation ----
lead = opt(f"{DATA}/s7_leaderboard.csv")
if lead is not None:
    print("LEADERBOARD (data/s7_leaderboard.csv):")
    print(lead.to_string(index=False))
    for _, r in lead.iterrows():   # verify banked rho against this notebook's own compute
        g = SV[SV.model == r.model]
        assert abs(spearmanr(g.level, g.p0a)[0] - r.rho_verbal) < 1e-3
    print("\ncross-check (rho_verbal vs recomputed): PASS")
else:
    print("s7_leaderboard.csv not present")


LEADERBOARD (data/s7_leaderboard.csv):
model  house_level  rho_verbal  rho_lo  rho_hi  med_delta  md_lo  md_hi content_share  cs_lo  cs_hi
  ace         4.35       0.328   0.167   0.476       0.10  -0.05    0.7     undefined  -8.07   9.47
  mgm         5.83       0.538   0.414   0.644       1.30   0.50    2.1          0.76   0.63   0.91
  mgs         5.81       0.388   0.247   0.512       2.05   0.80    2.4          0.76   0.47   0.99
 sa3s         5.80       0.384   0.223   0.525       1.60   0.55    2.2          0.82   0.64   1.15
  sao         6.04       0.504   0.367   0.619       1.30   0.55    2.0          0.59   0.50   0.69

cross-check (rho_verbal vs recomputed): PASS


In [10]:

# ---- Raw (pre-calibration) scale replicate of Table 1 (banked CIs) ----
if s7_cis is not None and (s7_cis.analysis == "rho_level_raw").any():
    sub = s7_cis[s7_cis.analysis.isin(["rho_level_raw", "delta_raw_mean", "delta_raw_median"])]
    piv = sub.pivot(index="model", columns="analysis", values=["point", "lo", "hi"]).round(3)
    print("RAW-SCALE REPLICATE (paper §4.4; ceiling-free):")
    for m in piv.index:
        f = lambda a: f'{piv.loc[m, ("point", a)]:+.2f} [{piv.loc[m, ("lo", a)]:+.2f},{piv.loc[m, ("hi", a)]:+.2f}]'
        print(f"  {m:5s} rho {f('rho_level_raw')}   d_mean {f('delta_raw_mean')}   d_median {f('delta_raw_median')}")
else:
    print("raw-scale CI rows not present in s7_bootstrap_cis.csv")


RAW-SCALE REPLICATE (paper §4.4; ceiling-free):
  ace   rho +0.34 [+0.18,+0.49]   d_mean +0.21 [-0.03,+0.43]   d_median +0.23 [-0.03,+0.62]
  mgm   rho +0.59 [+0.47,+0.68]   d_mean +1.86 [+1.47,+2.24]   d_median +1.86 [+1.42,+2.40]
  mgs   rho +0.41 [+0.27,+0.53]   d_mean +0.96 [+0.49,+1.42]   d_median +1.20 [+0.47,+1.66]
  sa3s  rho +0.41 [+0.25,+0.54]   d_mean +0.73 [+0.31,+1.13]   d_median +0.83 [+0.32,+1.33]
  sao   rho +0.54 [+0.40,+0.65]   d_mean +1.39 [+1.03,+1.75]   d_median +1.50 [+0.96,+1.82]


In [11]:

# ---- Fixed-anchor CLAP axis (activates when the axis CSVs are present) ----
axis_lm = opt(f"{DATA}/s7_axis_level_means.csv")
axis_pc = opt(f"{DATA}/s7_clap_axis.csv")
if axis_lm is not None:
    alm = axis_lm.set_index(["model", "phrasing", "level"])["axis"].unstack("level")
    print("axis means by level (verbal):")
    print(alm.xs("verbal", level="phrasing").round(4).to_string())
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for m in MODELS:
        ax.plot(LEVELS, alm.loc[(m, "verbal")].values, "-o", ms=4, label=m)
    ax.set_xlabel("requested energy level"); ax.set_ylabel("CLAP axis (sim_hi − sim_lo)")
    ax.set_xticks(LEVELS); ax.grid(alpha=0.25); ax.legend(fontsize=8)
    ax.set_title("Fixed-anchor CLAP axis, verbal (genre-matched anchors)")
    fig.tight_layout(); fig.savefig(f"{FIGS}/clap_axis_verbal.png", dpi=160, bbox_inches="tight")
    print(f"wrote {rel(FIGS)}/clap_axis_verbal.png")
    if s7_cis is not None and (s7_cis.analysis == "rho_level_axis").any():
        print("\nrho(level, axis) and axis droop (banked, 95% CIs):")
        for a in ["rho_level_axis", "droop_L9-L7_axis"]:
            for _, r in s7_cis[s7_cis.analysis == a].iterrows():
                print(f"  {a:18s} {r.model:5s} {r.scope:8s} {r.point:+.4f} [{r.lo:+.4f},{r.hi:+.4f}]")
else:
    print("CLAP-axis files pending (s7_axis_level_means.csv / s7_clap_axis.csv):")
    print("drop them into data/ from the canonical commit and re-run this notebook.")


axis means by level (verbal):
level       1       3       5       7       9
model                                        
ace   -0.1255 -0.1524 -0.1013 -0.0723 -0.0832
mgm   -0.0016  0.0060  0.0914  0.1060  0.1543
mgs   -0.0266 -0.0104  0.0703  0.1030  0.1131
sa3s  -0.0193 -0.0267  0.0788  0.0947  0.1013
sao   -0.0367  0.0016  0.0416  0.0445  0.0963


wrote rerun/step1/attempt2/out/figures/clap_axis_verbal.png

rho(level, axis) and axis droop (banked, 95% CIs):
  rho_level_axis     ace   numeric  -0.0112 [-0.1759,+0.1573]
  rho_level_axis     ace   verbal   +0.2743 [+0.1145,+0.4263]
  rho_level_axis     mgm   numeric  -0.0507 [-0.2073,+0.1076]
  rho_level_axis     mgm   verbal   +0.5961 [+0.4768,+0.6902]
  rho_level_axis     mgs   numeric  -0.0797 [-0.2385,+0.0764]
  rho_level_axis     mgs   verbal   +0.5460 [+0.4287,+0.6493]
  rho_level_axis     sa3s  numeric  +0.0113 [-0.1647,+0.1809]
  rho_level_axis     sa3s  verbal   +0.4502 [+0.2986,+0.5903]
  rho_level_axis     sao   numeric  +0.0208 [-0.1544,+0.1785]
  rho_level_axis     sao   verbal   +0.4623 [+0.3288,+0.5784]
  droop_L9-L7_axis   ace   verbal   -0.0110 [-0.0591,+0.0366]
  droop_L9-L7_axis   mgm   verbal   +0.0484 [+0.0058,+0.0906]
  droop_L9-L7_axis   mgs   verbal   +0.0101 [-0.0386,+0.0610]
  droop_L9-L7_axis   sa3s  verbal   +0.0065 [-0.0461,+0.0595]
  droop_L9-L7_axis  

## Banked statistics (S6, S7)
Regenerates every `data/s6_*.csv` and `data/s7_*.csv` bank into `TABLES`. Ported from `notebooks/scoring.ipynb` cells 13–14 and `notebooks/clap-axis.ipynb`; statistical logic unchanged.

In [12]:
# ============================================================
# S6 (v2) — BANKED STATISTICS: CIs, droop adjudication, phrasing contrast,
#      feature decomposition ("which knobs"), plausibility lens.
# Ported from notebooks/scoring.ipynb cell 13 (the canonical scoring run).
# Statistical logic is unchanged. Tier-1 adaptations only:
#   - input: DATA/scored_full_v2.csv (the file S6.0 banks) instead of
#     re-repairing scored_full.csv;
#   - S6.4: the banked per-clip feature cache DATA/features_verbal.csv and
#     the saved ridge DATA/arousal_index_ridge.json (scripts/fit_arousal_ridge.py)
#     instead of extracting audio and fitting on DEAM;
#   - every bank is written to TABLES (never to DATA).
# ============================================================
import json
from scipy.stats import spearmanr as _spearmanr

S6_B    = 2000   # bootstrap resamples (pre-committed in the study protocol)
S6_SEED = 0      # every bootstrap call re-seeds -> outputs identical across
                 # runs/sections regardless of whether S6.4 executes
S6_LEVELS = [1, 3, 5, 7, 9]

# Q7 writability bands (fixed before scoring; see paper §3).
# rock is an intensity-genre, not a Q7 palette: band below = H1 v2 catchy-zone
# (rock 5-8). ANALYST-SET (documented in the paper).
S6_BANDS = {'folk': (0.5, 5.5), 'edm': (4, 8), 'hiphop': (2, 7),
            'orchestral': (3, 7), 'rock': (5, 8)}

# ---------- S6.0 load ----------
S6 = pd.read_csv(f"{DATA}/scored_full_v2.csv")
print(f'S6.0  loaded {len(S6)} rows from {rel(DATA)}/scored_full_v2.csv')
assert len(S6) == 1500 and S6.file.is_unique, 'corpus shape drifted'

_SV = S6[S6.phrasing == 'verbal']
_SN = S6[S6.phrasing == 'numeric']

# ---------- bootstrap helpers (clip-level resampling throughout) ----------
def _ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

def _boot_rho(lv, sc, B=S6_B):
    rng = np.random.default_rng(S6_SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([_spearmanr(lv[i], sc[i])[0] for i in idx])

def _boot_stat(arrs, fn, B=S6_B):
    """arrs: list of 1-D (or 2-D, clips x cols) arrays resampled independently
    (clip level, within group); fn maps the resampled list -> scalar."""
    rng = np.random.default_rng(S6_SEED)
    out = np.empty(B)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    for b in range(B):
        out[b] = fn([a[ix[b]] for a, ix in zip(arrs, idxs)])
    return out

_ci_rows = []   # long-format bank: analysis, model, scope, point, lo, hi, note
def _bank(analysis, mdl, scope, point, boots, note=''):
    lo, hi = _ci(boots)
    _ci_rows.append(dict(analysis=analysis, model=mdl, scope=scope,
                         point=round(float(point), 4), lo=round(float(lo), 4),
                         hi=round(float(hi), 4), note=note))
    return lo, hi

_MODELS = sorted(S6.model.unique())
_grp = {(m, ph): g for (m, ph), g in S6.groupby(['model', 'phrasing'])}
_lvl = {(m, L): g for (m, L), g in _SV.groupby(['model', 'level'])}

# ---------- S6.1 per-level means + L9 droop adjudication ----------
print('\n' + '='*72 + '\nS6.1  PER-LEVEL MEANS (droop adjudication)\n' + '='*72)
_lm = (S6.groupby(['model','phrasing','level'])[['p0a','clap','lufs','p0a_n','clap_n']]
         .mean().round(3))
_lm.to_csv(f'{TABLES}/s6_level_means.csv')
for met in ['p0a', 'clap', 'lufs']:
    print(f'\nVERBAL mean {met} by level:')
    print(_lm.xs('verbal', level='phrasing')[met].unstack().round(2 if met!='clap' else 3).to_string())

print(f'\nDROOP PANEL — verbal L9 minus L7, 95% CI (clip-level, B={S6_B}):')
_droop = {}
for m in _MODELS:
    a7, a9 = _lvl[(m,7)], _lvl[(m,9)]
    for met in ['p0a', 'clap']:
        pt = a9[met].mean() - a7[met].mean()
        bo = _boot_stat([a7[met].values, a9[met].values], lambda xs: xs[1].mean()-xs[0].mean())
        lo, hi = _bank(f'droop_L9-L7_{met}', m, 'verbal', pt, bo)
        _droop[(m, met)] = (pt, lo, hi)
    dl = a9['lufs'].mean() - a7['lufs'].mean()
    print(f'  {m:5s} d_p0a {_droop[(m,"p0a")][0]:+.2f} [{_droop[(m,"p0a")][1]:+.2f},{_droop[(m,"p0a")][2]:+.2f}]'
          f'   d_clap {_droop[(m,"clap")][0]:+.3f} [{_droop[(m,"clap")][1]:+.3f},{_droop[(m,"clap")][2]:+.3f}]'
          f'   d_lufs {dl:+.2f} dB')

# ---------- S6.2 monotonicity + dynamic-range CIs (+ K4 share CIs) ----------
print('\n' + '='*72 + '\nS6.2  BOOTSTRAP CIs — rho, deltas (mean & median), K4 share\n' + '='*72)
print(f'{"model":<6}{"phr":<9}{"rho [95% CI]":<26}{"d(L9-L1) mean [CI]":<26}{"median [CI]"}')
for m in _MODELS:
    for ph in ['numeric', 'verbal']:
        g = _grp[(m, ph)]
        rho = _spearmanr(g.level, g.p0a)[0]
        rb = _boot_rho(g.level.values, g.p0a.values)
        rlo, rhi = _bank('rho_level_p0a', m, ph, rho, rb)
        a1 = g[g.level==1].p0a.values; a9 = g[g.level==9].p0a.values
        dm = a9.mean()-a1.mean()
        db = _boot_stat([a1, a9], lambda xs: xs[1].mean()-xs[0].mean())
        dmlo, dmhi = _bank('delta_L9-L1_mean', m, ph, dm, db)
        dmd = np.median(a9)-np.median(a1)
        dbmd = _boot_stat([a1, a9], lambda xs: np.median(xs[1])-np.median(xs[0]))
        dmdlo, dmdhi = _bank('delta_L9-L1_median', m, ph, dmd, dbmd)
        print(f'{m:<6}{ph:<9}{rho:+.3f} [{rlo:+.3f},{rhi:+.3f}]   '
              f'{dm:+.2f} [{dmlo:+.2f},{dmhi:+.2f}]   {dmd:+.2f} [{dmdlo:+.2f},{dmdhi:+.2f}]')

print('\nK4 content_share = d_norm/d_raw (verbal, paired clip resample):')
for m in _MODELS:
    A1, A9 = _lvl[(m,1)][['p0a','p0a_n']].values, _lvl[(m,9)][['p0a','p0a_n']].values
    raw_d  = A9[:,0].mean()-A1[:,0].mean(); norm_d = A9[:,1].mean()-A1[:,1].mean()
    sb = _boot_stat([A1, A9], content_share)      # content_share() defined in the setup cell
    rb = _boot_stat([A1, A9], lambda xs: xs[1][:,0].mean()-xs[0][:,0].mean())
    unstable = (_ci(rb)[0] <= 0 <= _ci(rb)[1])
    note = 'UNSTABLE: raw-delta CI covers 0; ratio meaningless' if unstable else ''
    lo, hi = _bank('K4_content_share', m, 'verbal', norm_d/raw_d, sb, note)
    tail = f'  << {note}' if unstable else ''
    print(f'  {m:5s} share {norm_d/raw_d:+.2f} [{lo:+.2f},{hi:+.2f}]  (raw d {raw_d:+.2f}, norm d {norm_d:+.2f}){tail}')

# ---------- S6.3 phrasing contrast (verbal - numeric) ----------
print('\n' + '='*72 + '\nS6.3  PHRASING EFFECT — verbal minus numeric, per model\n' + '='*72)
for m in _MODELS:
    gn, gv = _grp[(m,'numeric')], _grp[(m,'verbal')]
    # NOTE: each _boot_rho call re-seeds at 0, so with equal arm sizes the two
    # arms draw the same index matrix (paired by row position), not
    # independent resamples. Kept as-is: it is how the banked CIs were made.
    rvb = _boot_rho(gv.level.values, gv.p0a.values); rnb = _boot_rho(gn.level.values, gn.p0a.values)
    drho = _spearmanr(gv.level, gv.p0a)[0] - _spearmanr(gn.level, gn.p0a)[0]
    rlo, rhi = _bank('phrasing_drho', m, 'verbal-numeric', drho, rvb - rnb)
    arrs = [gn[gn.level==1].p0a.values, gn[gn.level==9].p0a.values,
            gv[gv.level==1].p0a.values, gv[gv.level==9].p0a.values]
    dd = (arrs[3].mean()-arrs[2].mean()) - (arrs[1].mean()-arrs[0].mean())
    ddb = _boot_stat(arrs, lambda xs: (xs[3].mean()-xs[2].mean())-(xs[1].mean()-xs[0].mean()))
    dlo, dhi = _bank('phrasing_ddelta', m, 'verbal-numeric', dd, ddb)
    print(f'  {m:5s} d_rho {drho:+.3f} [{rlo:+.3f},{rhi:+.3f}]   d_delta {dd:+.2f} [{dlo:+.2f},{dhi:+.2f}]')

# ---------- S6.4 feature block: raw ridge + which-knobs decomposition ----------
print('\n' + '='*72 + '\nS6.4  FEATURE DECOMPOSITION (banked features + saved ridge)\n' + '='*72)
_CLUSTERS = {'loudness': ['loudness_mean','loudness_std'],
             'noisiness': ['zcr_mean','zcr_std'],
             'tonality':  ['hnr_mean','hnr_std']}
SIX = ['loudness_mean','loudness_std','brightness_mean','brightness_std','flux_mean',
       'flux_std','zcr_mean','zcr_std','sharpness_mean','sharpness_std','hnr_mean','hnr_std']
_ridge = json.load(open(f'{DATA}/arousal_index_ridge.json'))
CORE = _ridge['features']
class _Fitted:   # stand-ins for the fitted sklearn steps; attribute names kept
    pass
_sc, _rg = _Fitted(), _Fitted()
_sc.mean_, _sc.scale_ = np.array(_ridge['scaler_mean']), np.array(_ridge['scaler_scale'])
_rg.coef_, _rg.intercept_ = np.array(_ridge['coef']), float(_ridge['intercept'])
_feat = pd.read_csv(f'{DATA}/features_verbal.csv')
# The canonical run computed these features in memory, where librosa returns
# float32 for rms (loudness), onset strength (flux) and spectral flatness
# (sharpness, and hnr derived from it); centroid and zcr are float64. The CSV
# stores each value's shortest decimal form, so cast those columns back to
# float32 to recover the exact in-memory values (and float32 group means).
_F32 = ['loudness_mean','loudness_std','flux_mean','flux_std',
        'sharpness_mean','sharpness_std','hnr_mean','hnr_std']
for c6 in _F32: _feat[c6] = _feat[c6].astype(np.float32)
_need = _SV.file.tolist()
_feat = _feat[_feat.file.isin(_need)]
cov = len(_feat)/len(_need)
print(f'  coverage: {len(_feat)}/{len(_need)} verbal clips ({cov:.0%})')
assert cov >= 0.90, 'feature coverage < 90%'
_fm = _feat.merge(_SV[['file','model','level']], on='file')
Xz = (_fm[CORE].values - _sc.mean_) / _sc.scale_
_fm['raw_score'] = Xz @ _rg.coef_ + _rg.intercept_
_rawlvl = _fm.groupby(['model','level']).raw_score.mean().unstack()
print('\n  RAW ridge score (pre-calibration), verbal means by level:')
print(_rawlvl.round(2).to_string())
_rawlvl.to_csv(f'{TABLES}/s6_raw_ridge_level_means.csv')
print('  droop on the raw score (second adjudicator):')
for m in _MODELS:
    g7 = _fm[(_fm.model==m)&(_fm.level==7)].raw_score.values
    g9 = _fm[(_fm.model==m)&(_fm.level==9)].raw_score.values
    if not len(g7) or not len(g9): continue
    pt = g9.mean()-g7.mean()
    bo = _boot_stat([g7, g9], lambda xs: xs[1].mean()-xs[0].mean())
    lo, hi = _bank('droop_L9-L7_rawridge', m, 'verbal', pt, bo)
    print(f'    {m:5s} d_raw(L9-L7) {pt:+.2f} [{lo:+.2f},{hi:+.2f}]')
# which knobs: exact linear split of the raw-score delta, L9-L1
print('\n  WHICH KNOBS — cluster contributions to raw-score delta (L9-L1, verbal):')
_dec = []
for m in _MODELS:
    mu = _fm[_fm.model==m].groupby('level')[CORE].mean()
    if not {1,9} <= set(mu.index): continue
    dz = (mu.loc[9]-mu.loc[1]).values / _sc.scale_
    contrib = pd.Series(_rg.coef_ * dz, index=CORE)
    tot = contrib.sum()
    row = {'model': m, 'd_raw_total': round(tot,3)}
    cells = []
    for cname, feats in _CLUSTERS.items():
        cval = contrib[feats].sum()
        row[f'{cname}_contrib'] = round(cval,3)
        row[f'{cname}_share'] = round(cval/tot,3) if abs(tot) > 1e-9 else np.nan
        cells.append(f'{cval:+.2f} ({cval/tot:+.0%})' if abs(tot) > 1e-9 else f'{cval:+.2f} ( -- )')
    for f6 in SIX: row[f'dfeat_{f6}'] = (_fm[_fm.model==m].groupby("level")[f6].mean().loc[9]
                                         - _fm[_fm.model==m].groupby("level")[f6].mean().loc[1])
    _dec.append(row)
    print(f'  {m:<6}{tot:+7.2f} | ' + ' | '.join(f'{c:>16}' for c in cells))
pd.DataFrame(_dec).to_csv(f'{TABLES}/s6_feature_decomp.csv', index=False)

# ---------- S6.5 plausibility lens: Q7 edge cells ----------
print('\n' + '='*72 + '\nS6.5  PLAUSIBILITY LENS — Q7 bands, edge cells (verbal)\n' + '='*72)
_ec = _SV.groupby(['model','genre_slug','level']).p0a.mean().round(2).unstack()
def _inb(g, L): lo, hi = S6_BANDS[g]; return lo <= L <= hi
_marked = _ec.copy().astype(object)
for g in _marked.index.get_level_values('genre_slug').unique():
    for L in S6_LEVELS:
        if not _inb(g, L):
            _marked.loc[(slice(None), g), L] = _marked.loc[(slice(None), g), L].map(lambda x: f'{x}*')
print('\n  mean p0a per cell ("*" = requested level OUTSIDE genre band; the CSV is unmarked):')
print(_marked.to_string())
_ec.to_csv(f'{TABLES}/s6_edge_cells.csv')
print('\n  step-gain contrast: per-unit p0a gain on IN-band vs OUT/edge steps')
_steps = [(1,3),(3,5),(5,7),(7,9)]
for scope_m in _MODELS + ['ALL']:
    sub = _SV if scope_m == 'ALL' else _SV[_SV.model == scope_m]
    cells = {(g,L): gg.p0a.values for (g,L), gg in sub.groupby(['genre_slug','level'])}
    keys = sorted(cells); arrs = [cells[k] for k in keys]; pos = {k:i for i,k in enumerate(keys)}
    def _gap(xs, which):
        gains = []
        for g in S6_BANDS:
            for a,b in _steps:
                inb = _inb(g,a) and _inb(g,b)
                if inb == which and (g,a) in pos and (g,b) in pos:
                    gains.append((xs[pos[(g,b)]].mean()-xs[pos[(g,a)]].mean())/(b-a))
        return float(np.mean(gains))
    gin, gout = _gap(arrs, True), _gap(arrs, False)
    bo = _boot_stat(arrs, lambda xs: _gap(xs, True)-_gap(xs, False))
    lo, hi = _bank('edge_gain_in_minus_out', scope_m, 'verbal', gin-gout, bo)
    print(f'  {scope_m:5s} in {gin:+.3f}  out {gout:+.3f}  diff {gin-gout:+.3f} [{lo:+.3f},{hi:+.3f}]')

# ---------- S6.6 bank ----------
pd.DataFrame(_ci_rows).to_csv(f'{TABLES}/s6_bootstrap_cis.csv', index=False)
print(f'\nS6.6  BANKED to {rel(TABLES)}/: s6_level_means.csv, s6_bootstrap_cis.csv, s6_edge_cells.csv,')
print('      s6_raw_ridge_level_means.csv, s6_feature_decomp.csv.  S6 complete.')


S6.0  loaded 1500 rows from data/scored_full_v2.csv

S6.1  PER-LEVEL MEANS (droop adjudication)

VERBAL mean p0a by level:
level     1     3     5     7     9
model                              
ace    4.16  4.00  4.33  4.90  4.35
mgm    5.16  5.22  5.95  6.01  6.44
mgs    4.76  5.10  5.55  6.11  5.69
sa3s   4.95  4.90  5.92  6.08  5.78
sao    5.19  5.61  5.76  6.09  6.32

VERBAL mean clap by level:
level      1      3      5      7      9
model                                   
ace    0.064  0.068  0.115  0.233  0.152
mgm    0.168  0.199  0.272  0.280  0.373
mgs    0.066  0.092  0.208  0.303  0.257
sa3s   0.136  0.130  0.314  0.320  0.318
sao    0.158  0.207  0.246  0.294  0.361

VERBAL mean lufs by level:
level      1      3      5      7      9
model                                   
ace   -18.74 -19.26 -18.98 -18.61 -19.12
mgm   -19.14 -18.45 -20.30 -17.51 -16.84
mgs   -21.64 -20.40 -18.92 -16.81 -19.73
sa3s  -18.92 -18.72 -16.03 -16.68 -17.53
sao   -19.93 -18.31 -18.71 -17.02 -1

ace   numeric  +0.070 [-0.089,+0.228]   +0.09 [-0.21,+0.38]   +0.10 [-0.30,+0.40]


ace   verbal   +0.328 [+0.167,+0.476]   +0.18 [-0.12,+0.47]   +0.10 [-0.05,+0.70]


mgm   numeric  +0.038 [-0.134,+0.202]   +0.00 [-0.41,+0.44]   +0.10 [-1.00,+0.90]


mgm   verbal   +0.538 [+0.414,+0.644]   +1.28 [+0.91,+1.63]   +1.30 [+0.50,+2.10]


mgs   numeric  +0.041 [-0.127,+0.197]   +0.08 [-0.36,+0.52]   +0.30 [-0.50,+1.20]


... (11 outputs left out)

  ace   in +0.201  out -0.096  diff +0.297 [+0.185,+0.412]


  mgm   in +0.283  out +0.077  diff +0.206 [+0.052,+0.344]
  mgs   in +0.196  out +0.064  diff +0.132 [-0.031,+0.301]


  sa3s  in +0.278  out -0.013  diff +0.291 [+0.179,+0.400]
  sao   in +0.157  out +0.131  diff +0.026 [-0.074,+0.128]


  ALL   in +0.223  out +0.033  diff +0.190 [+0.096,+0.279]

S6.6  BANKED to rerun/step1/attempt2/out/tables/: s6_level_means.csv, s6_bootstrap_cis.csv, s6_edge_cells.csv,
      s6_raw_ridge_level_means.csv, s6_feature_decomp.csv.  S6 complete.


In [13]:
# ============================================================
# S7 — raw-scale CIs, sensitivity banks, leaderboard, CLAP-axis stats.
# Ported from notebooks/scoring.ipynb cell 14 (S7.2-S7.6) and
# notebooks/clap-axis.ipynb (S7A.2 stats + S7A.3 merge). Statistical logic
# unchanged. Tier-1 adaptations only: the per-clip axis scores are read from
# DATA/s7_clap_axis.csv (producing them needs the audio: Tier 2), the raw
# ridge scores come from S6.4 above, and banks go to TABLES.
# Every bootstrap re-seeds at 0 -> digit-identical across runs.
# ============================================================
from scipy.stats import spearmanr as _sp7

S7_B, S7_SEED = 2000, 0
S7 = pd.read_csv(f'{DATA}/scored_full_v2.csv')
assert len(S7) == 1500
_SV7 = S7[S7.phrasing == 'verbal']
_M7 = sorted(S7.model.unique())

def _ci7(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return np.percentile(a, 2.5), np.percentile(a, 97.5)

def _brho7(lv, sc, B=S7_B):
    rng = np.random.default_rng(S7_SEED)
    lv, sc = np.asarray(lv), np.asarray(sc); n = len(lv)
    idx = rng.integers(0, n, (B, n))
    return np.array([_sp7(lv[i], sc[i])[0] for i in idx])

def _bstat7(arrs, fn, B=S7_B):
    rng = np.random.default_rng(S7_SEED)
    idxs = [rng.integers(0, len(a), (B, len(a))) for a in arrs]
    return np.array([fn([a[ix[b]] for a, ix in zip(arrs, idxs)]) for b in range(B)])

_rows7 = []
def _bank7(analysis, mdl, scope, point, boots, note=''):
    lo, hi = _ci7(boots)
    _rows7.append(dict(analysis=analysis, model=mdl, scope=scope,
                       point=round(float(point), 4), lo=round(float(lo), 4),
                       hi=round(float(hi), 4), note=note))
    return lo, hi

# ---------- S7.2 raw (pre-calibration) scale CIs, verbal ----------
print('='*72 + '\nS7.2  RAW-SCALE CIs (verbal)\n' + '='*72)
_fm7 = _feat.merge(_SV7[['file','model','level']], on='file')     # S6.4 feature cache
_fm7['raw'] = ((_fm7[CORE].values - _sc.mean_) / _sc.scale_) @ _rg.coef_ + _rg.intercept_
print(f'  {"model":<6}{"rho_raw [CI]":<24}{"d(9-1) mean [CI]":<24}{"median [CI]"}')
for m in _M7:
    g = _fm7[_fm7.model==m]
    r = _sp7(g.level, g.raw)[0]
    rlo, rhi = _bank7('rho_level_raw', m, 'verbal', r, _brho7(g.level.values, g.raw.values))
    a1, a9 = g[g.level==1].raw.values, g[g.level==9].raw.values
    dm = a9.mean()-a1.mean()
    dlo, dhi = _bank7('delta_raw_mean', m, 'verbal', dm,
                      _bstat7([a1,a9], lambda xs: xs[1].mean()-xs[0].mean()))
    dmd = np.median(a9)-np.median(a1)
    mlo, mhi = _bank7('delta_raw_median', m, 'verbal', dmd,
                      _bstat7([a1,a9], lambda xs: np.median(xs[1])-np.median(xs[0])))
    print(f'  {m:<6}{r:+.3f} [{rlo:+.3f},{rhi:+.3f}]   {dm:+.2f} [{dlo:+.2f},{dhi:+.2f}]'
          f'    {dmd:+.2f} [{mlo:+.2f},{mhi:+.2f}]')

# ---------- S7.3 plausibility contrast, rock excluded ----------
print('\n' + '='*72 + '\nS7.3  BAND CONTRAST, ROCK EXCLUDED\n' + '='*72)
_B7 = {'folk':(0.5,5.5),'edm':(4,8),'hiphop':(2,7),'orchestral':(3,7),'rock':(5,8)}
_steps7 = [(1,3),(3,5),(5,7),(7,9)]
def _inb7(g,L): lo,hi=_B7[g]; return lo<=L<=hi
def _contrast7(sub, genres):
    cells = {(g,L):gg.p0a.values for (g,L),gg in sub.groupby(['genre_slug','level']) if g in genres}
    keys = sorted(cells); arrs=[cells[k] for k in keys]; pos={k:i for i,k in enumerate(keys)}
    def gap(xs, which):
        gains=[(xs[pos[(g,b)]].mean()-xs[pos[(g,a)]].mean())/(b-a)
               for g in genres for a,b in _steps7
               if (_inb7(g,a) and _inb7(g,b))==which and (g,a) in pos]
        return float(np.mean(gains))
    pt = gap(arrs,True)-gap(arrs,False)
    return pt, _bstat7(arrs, lambda xs: gap(xs,True)-gap(xs,False))
_ng = [g for g in _B7 if g != 'rock']
for scope in ['ALL'] + _M7:
    sub = _SV7 if scope=='ALL' else _SV7[_SV7.model==scope]
    pt, bo = _contrast7(sub, _ng)
    lo, hi = _bank7('edge_gain_norock', scope, 'verbal', pt, bo)
    print(f'  {scope:5s} in-out diff {pt:+.3f} [{lo:+.3f},{hi:+.3f}]')

# ---------- S7.4 family contrasts (scale, era) ----------
print('\n' + '='*72 + '\nS7.4  FAMILY CONTRASTS (verbal rho)\n' + '='*72)
def _rho_arm(m):
    g = _SV7[_SV7.model==m]; return g.level.values, g.p0a.values
for name, (m1, m2) in {'scale_mgm-mgs':('mgm','mgs'), 'era_sa3s-sao':('sa3s','sao')}.items():
    l1,s1 = _rho_arm(m1); l2,s2 = _rho_arm(m2)
    pt = _sp7(l1,s1)[0] - _sp7(l2,s2)[0]
    lo, hi = _bank7('family_drho', name, 'verbal', pt, _brho7(l1,s1)-_brho7(l2,s2))
    print(f'  {name}: d_rho {pt:+.3f} [{lo:+.3f},{hi:+.3f}]')

# ---------- S7.5 skew exhibit bank + S7.6 leaderboard ----------
_SV7[_SV7.level.isin([1,9])][['model','genre_slug','level','seed','p0a']] \
    .to_csv(f'{TABLES}/s7_skew_L1L9.csv', index=False)
_lead = []
for m in _M7:
    g = _SV7[_SV7.model==m]; gn = S7[(S7.model==m)&(S7.phrasing=='numeric')]
    r = _sp7(g.level, g.p0a)[0]; rb = _brho7(g.level.values, g.p0a.values)
    a1,a9 = g[g.level==1].p0a.values, g[g.level==9].p0a.values
    A1 = g[g.level==1][['p0a','p0a_n']].values; A9 = g[g.level==9][['p0a','p0a_n']].values
    md = np.median(a9)-np.median(a1)
    mdb = _bstat7([a1,a9], lambda xs: np.median(xs[1])-np.median(xs[0]))
    raw_d = a9.mean()-a1.mean()
    rdb = _bstat7([a1,a9], lambda xs: xs[1].mean()-xs[0].mean())
    _unstable = _ci7(rdb)[0] <= 0 <= _ci7(rdb)[1]
    cs = np.nan if _unstable else (A9[:,1].mean()-A1[:,1].mean())/raw_d
    csb = _bstat7([A1,A9], content_share)
    _lead.append(dict(model=m, house_level=round(gn.p0a.mean(),2),
                      rho_verbal=round(r,3), rho_lo=round(_ci7(rb)[0],3), rho_hi=round(_ci7(rb)[1],3),
                      med_delta=round(md,2), md_lo=round(_ci7(mdb)[0],2), md_hi=round(_ci7(mdb)[1],2),
                      content_share=round(cs,2) if not np.isnan(cs) else 'undefined',
                      cs_lo=round(_ci7(csb)[0],2), cs_hi=round(_ci7(csb)[1],2)))
pd.DataFrame(_lead).to_csv(f'{TABLES}/s7_leaderboard.csv', index=False)

# ---------- S7A.2 fixed-anchor CLAP axis stats (from notebooks/clap-axis.ipynb) ----------
print('\n' + '='*72 + '\nS7A.2  AXIS STATS\n' + '='*72)
_ax = pd.read_csv(f'{DATA}/s7_clap_axis.csv')
for c in ['sim_lo','sim_hi','axis']: _ax[c] = pd.to_numeric(_ax[c])
assert len(_ax) / 1500 >= 0.90, 'axis coverage < 90%'
_A = S7.merge(_ax, on='file')
_rows = []
def _bank_ax(analysis, mdl, scope, point, boots, note=''):
    lo, hi = _ci7(boots)
    _rows.append(dict(analysis=analysis, model=mdl, scope=scope,
                      point=round(float(point), 4), lo=round(float(lo), 4),
                      hi=round(float(hi), 4), note=note))
    return lo, hi
_A.groupby(['model','phrasing','level'])['axis'].mean().round(4).to_csv(f'{TABLES}/s7_axis_level_means.csv')
print('  rho(level, axis) with 95% CI:')
for m in _M7:
    for ph in ['numeric','verbal']:
        g = _A[(_A['model']==m)&(_A['phrasing']==ph)]
        r = _sp7(g['level'], g['axis'])[0]
        lo, hi = _bank_ax('rho_level_axis', m, ph, r, _brho7(g['level'].values, g['axis'].values))
        print(f'    {m:5s} {ph:8s} {r:+.3f} [{lo:+.3f},{hi:+.3f}]')
print('  axis droop (verbal L9 - L7):')
for m in _M7:
    g7 = _A[(_A['model']==m)&(_A['phrasing']=='verbal')&(_A['level']==7)]['axis'].values
    g9 = _A[(_A['model']==m)&(_A['phrasing']=='verbal')&(_A['level']==9)]['axis'].values
    pt = g9.mean()-g7.mean()
    lo, hi = _bank_ax('droop_L9-L7_axis', m, 'verbal', pt,
                      _bstat7([g7,g9], lambda xs: xs[1].mean()-xs[0].mean()))
    print(f'    {m:5s} {pt:+.4f} [{lo:+.4f},{hi:+.4f}]')

# ---------- S7A.3 merge + bank (axis rows appended after the S7 rows) ----------
_base = pd.DataFrame(_rows7)
_base = _base[~_base['analysis'].isin(['rho_level_axis','droop_L9-L7_axis'])]
pd.concat([_base, pd.DataFrame(_rows)], ignore_index=True) \
  .to_csv(f'{TABLES}/s7_bootstrap_cis.csv', index=False)
print(f'\nS7 BANKED to {rel(TABLES)}/: s7_bootstrap_cis.csv ({len(_base)} S7 rows + {len(_rows)} axis rows),')
print('  s7_axis_level_means.csv, s7_skew_L1L9.csv, s7_leaderboard.csv.  S7 complete.')


S7.2  RAW-SCALE CIs (verbal)
  model rho_raw [CI]            d(9-1) mean [CI]        median [CI]


  ace   +0.342 [+0.177,+0.494]   +0.21 [-0.03,+0.43]    +0.23 [-0.03,+0.62]


  mgm   +0.586 [+0.468,+0.684]   +1.86 [+1.47,+2.24]    +1.86 [+1.41,+2.40]


  mgs   +0.406 [+0.268,+0.525]   +0.96 [+0.49,+1.42]    +1.20 [+0.47,+1.66]


  sa3s  +0.406 [+0.247,+0.543]   +0.73 [+0.31,+1.13]    +0.83 [+0.32,+1.33]


  sao   +0.537 [+0.403,+0.648]   +1.39 [+1.03,+1.75]    +1.50 [+0.96,+1.82]

S7.3  BAND CONTRAST, ROCK EXCLUDED
  ALL   in-out diff +0.217 [+0.111,+0.321]


... (12 outputs left out)

    sa3s  numeric  +0.011 [-0.165,+0.181]


    sa3s  verbal   +0.450 [+0.299,+0.590]


    sao   numeric  +0.021 [-0.154,+0.178]


    sao   verbal   +0.462 [+0.329,+0.578]
  axis droop (verbal L9 - L7):
    ace   -0.0110 [-0.0591,+0.0366]
    mgm   +0.0484 [+0.0058,+0.0906]
    mgs   +0.0101 [-0.0386,+0.0610]
    sa3s  +0.0065 [-0.0461,+0.0595]
    sao   +0.0518 [+0.0094,+0.0959]

S7 BANKED to rerun/step1/attempt2/out/tables/: s7_bootstrap_cis.csv (23 S7 rows + 15 axis rows),
  s7_axis_level_means.csv, s7_skew_L1L9.csv, s7_leaderboard.csv.  S7 complete.


## Notes
- Arousal Index and LUFS values are bit-reproducible across scoring runs. Own-prompt CLAP (`clap`, `clap_n` columns) varied between historical runs because the CLAP feature extractor crops audio longer than 10 s at random with numpy's global RNG; `notebooks/scoring.ipynb` now seeds that RNG before every CLAP call. The released values are from the run that produced `data/`; the fixed-anchor CLAP axis (`data/s7_*.csv`) was deterministic throughout.
- The S6.4/S7.2 raw-ridge numbers use `data/arousal_index_ridge.json`, re-fit from DEAM with `scripts/fit_arousal_ridge.py`; they can differ from the banked values in the last binary digit (~1e-15) depending on the linear-algebra build.
- Deep reproduction (re-scoring audio from scratch) uses `notebooks/scoring.ipynb` and `notebooks/clap-axis.ipynb` (Kaggle, GPU).